# Save a small AOPWiki dataset

Keep Phenobarbital, its stressor annotations, their AOPs, and the AOPs' key events as RDF. We retrieve selected records, not the full dump.

Run the [thyroid investigation](AOPWiki_hydration.ipynb) first to create `thyroid-session.json`. Its saved schema supplies the same Python classes. This notebook makes a few new endpoint requests, then works from the saved subset.

In [1]:
import json
from pathlib import Path

from rdflib import Graph
from rdfsolve import MinedSchema

session = json.loads(Path("thyroid-session.json").read_text())
schema = MinedSchema.from_dict(session["schema"])
client = schema.client(max_subjects=500)
Chemical = client.model("ChemicalEntity")
Stressor = client.model("Stressor")
AOP = client.model("AdverseOutcomePathway")
KeyEvent = client.model("KeyEvent")

## Choose the neighbourhood

Request both names and the links we want to save. Following a link retrieves records; it does not automatically load every field.

In [2]:
with client.step("Read the Phenobarbital neighbourhood"):
    chemicals = client.search(Chemical, "Phenobarbital", fields=["title", "identifier", "exactmatch"])
    stressors = client.follow(
        chemicals, "has_chemical_entity", Stressor, inverse=True,
        fields=["title", "has_chemical_entity"],
    )
    aops = client.follow(
        stressors, "c54571", AOP, inverse=True,
        fields=["title", "c54571", "has_key_event"],
    )
    events = client.follow(aops, "has_key_event", KeyEvent, fields=["title"])

client.table(aops, ["title"])

,uri,title
0,https://identifiers.org/aop/107,Constitutive androstane receptor activation le...
1,https://identifiers.org/aop/162,Enhanced hepatic clearance of thyroid hormones...


## Write the records as RDF

Each object writes its populated fields using the predicates in its model. Retrieved literals keep their original text, language, and datatype. The output also keeps the source's returned types; it does not assert that mapped records are identical.

In [3]:
records = chemicals + stressors + aops + events
subset = Graph()
for record in records:
    subset += record.to_graph()

subset.serialize("phenobarbital-subset.ttl", format="turtle")
client.save_session("phenobarbital-session.json")
client.close()
print(f"Saved {len(records)} records as {len(subset)} RDF statements")

Saved 16 records as 63 RDF statements


This is a selected neighbourhood, not a complete copy of these resources. Some identifier and stressor links point outside it. Those links remain valid even when their targets were not downloaded.

The file is one RDF graph. It does not preserve separate named-graph membership; the session records which graph was queried.

## Use the subset without the endpoint

Read the file and use the same generated API locally. No network request is needed for this step.

In [4]:
saved = Graph().parse("phenobarbital-subset.ttl", format="turtle")
with schema.client(saved, graph_uris=[]) as local:
    found = local.search(local.model("ChemicalEntity"), "Phenobarbital", fields=["title", "identifier"])
    display(local.table(found, ["title", "identifier"]))

assert set(saved) == set(subset)

,uri,title,identifier
0,https://identifiers.org/cas/50-06-6,Phenobarbital,https://identifiers.org/cas/50-06-6


## What can be written back?

Direct fields can be written as triples. A reversed single predicate can also be written in its original direction. A multi-step path returns terminal values, not the intermediate statements needed to recreate that path: `to_graph()` raises instead of inventing them. Select direct fields with `record.to_graph(fields=["title"])`, or retrieve the intermediate records as above.

If you edit a retrieved field, its old `rdf_terms` entry must be updated or removed before exporting. This prevents an old lexical value from silently replacing your edit.

For future mapped views, keep source records and their own field definitions distinct. A class mapping alone does not justify merging instance IRIs or adding `owl:sameAs`. Merged cross-dataset models are not implemented here.